# CS801 Part 1 - Customer Segmentation: quantitative clustering analysis

This notebook develops a quantitative clustering workflow for customer segmentation. The analysis focuses on defensible preprocessing, explicit missingness handling, statistically motivated outlier treatment, nominal-category encoding, PCA explained variance, cluster validation, and formal tests for cluster profiles.

Cell labels use the prefix `MW-METHOD #xx` so the accompanying report can reference stable points even if Jupyter execution numbers change.

In [ ]:
# MW-METHOD #01 - Imports and reproducibility settings
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import kruskal, chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.cs801_utils import safe_read_csv, iqr_bounds, cramers_v, benjamini_hochberg

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

In [ ]:
# MW-METHOD #02 - Data loading with a reproducible project path convention
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PATH = DATA_DIR / "marketing_campaign.csv"

raw = safe_read_csv(DATA_PATH, sep="	")
print(f"Rows: {raw.shape[0]:,}; columns: {raw.shape[1]:,}")
display(raw.head())
display(raw.info())

## Objective and success criteria

This case study treats customer segmentation as an unsupervised learning problem with measurable success criteria:

1. create customer segments from demographic and purchasing variables;
2. justify preprocessing decisions statistically;
3. select the dimensionality-reduction and clustering settings with metrics, not only visual judgement;
4. test whether the final segments are meaningfully different on business-relevant variables.

In [ ]:
# MW-METHOD #03 - Missingness audit rather than listwise deletion
missing = (
    raw.isna()
       .mean()
       .mul(100)
       .rename("missing_percent")
       .to_frame()
       .sort_values("missing_percent", ascending=False)
)
display(missing[missing["missing_percent"] > 0])

# Check whether Income missingness is independent of major categorical variables.
# This does not prove MCAR, but it is stronger than silently dropping the rows.
missing_tests = []
if "Income" in raw.columns:
    income_missing = raw["Income"].isna()
    for col in ["Education", "Marital_Status", "Kidhome", "Teenhome", "Response"]:
        if col in raw.columns:
            table = pd.crosstab(income_missing, raw[col])
            if table.shape[0] > 1 and table.shape[1] > 1:
                chi2, p_value, dof, expected = chi2_contingency(table)
                missing_tests.append({
                    "missing_variable": "Income",
                    "compared_with": col,
                    "chi2": chi2,
                    "p_value": p_value,
                    "cramers_v": cramers_v(table),
                })
missing_tests = pd.DataFrame(missing_tests)
if not missing_tests.empty:
    missing_tests["p_adj_bh"] = benjamini_hochberg(missing_tests["p_value"])
    display(missing_tests.sort_values("p_adj_bh"))

print("Decision: keep rows and handle missing Income inside the preprocessing pipeline with median imputation + missingness indicators.")

In [ ]:
# MW-METHOD #04 - Feature engineering with a data-derived reference date
customers = raw.copy()
customers["Dt_Customer"] = pd.to_datetime(customers["Dt_Customer"], format="%d-%m-%Y", errors="coerce")
reference_date = customers["Dt_Customer"].max()

spend_cols_original = ["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
spend_rename = {
    "MntWines": "Wines",
    "MntFruits": "Fruits",
    "MntMeatProducts": "Meat",
    "MntFishProducts": "Fish",
    "MntSweetProducts": "Sweets",
    "MntGoldProds": "Gold",
}
customers = customers.rename(columns=spend_rename)
spend_cols = list(spend_rename.values())

customers["Customer_For_Days"] = (reference_date - customers["Dt_Customer"]).dt.days
customers["Age"] = reference_date.year - customers["Year_Birth"]
customers["Spent"] = customers[spend_cols].sum(axis=1)
customers["Children"] = customers["Kidhome"] + customers["Teenhome"]
customers["Living_With"] = customers["Marital_Status"].replace({
    "Married": "Partner", "Together": "Partner", "Single": "Alone", "Divorced": "Alone",
    "Widow": "Alone", "Alone": "Alone", "Absurd": "Alone", "YOLO": "Alone",
})
customers["Family_Size"] = customers["Living_With"].map({"Alone": 1, "Partner": 2}).fillna(1) + customers["Children"]
customers["Is_Parent"] = (customers["Children"] > 0).astype(int)
customers["Education_Group"] = customers["Education"].replace({
    "Basic": "Undergraduate", "2n Cycle": "Undergraduate",
    "Graduation": "Graduate", "Master": "Postgraduate", "PhD": "Postgraduate",
})
customers["Total_Promos"] = customers[[c for c in ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5", "Response"] if c in customers]].sum(axis=1)

display(customers[["Education_Group", "Living_With", "Income", "Age", "Spent", "Customer_For_Days", "Children", "Family_Size"]].head())

In [ ]:
# MW-METHOD #05 - Statistically justified outlier treatment using IQR bounds
# Hard-coded caps can be difficult to defend. Here we report IQR bounds and winsorise
# extreme values instead of deleting observations by arbitrary thresholds.
outlier_cols = ["Income", "Age", "Spent"]
outlier_summary = []
customers_clean = customers.copy()

for col in outlier_cols:
    if col in customers_clean.columns:
        lower, upper = iqr_bounds(customers_clean[col], k=1.5)
        outlier_flag = ((customers_clean[col] < lower) | (customers_clean[col] > upper)).astype(int)
        customers_clean[f"{col}_IQR_outlier"] = outlier_flag
        customers_clean[col] = customers_clean[col].clip(lower=lower, upper=upper)
        outlier_summary.append({
            "feature": col,
            "iqr_lower": lower,
            "iqr_upper": upper,
            "n_outliers": int(outlier_flag.sum()),
            "outlier_percent": 100 * outlier_flag.mean(),
        })

outlier_summary = pd.DataFrame(outlier_summary)
display(outlier_summary)
print("Decision: use winsorised values for clustering and retain outlier flags as potential signal.")

In [ ]:
# MW-METHOD #06 - Preprocessing pipeline: imputation, robust scaling, one-hot encoding
numeric_features = [
    "Income", "Recency", "Customer_For_Days", "Age", "Children", "Family_Size", "Is_Parent",
    "Kidhome", "Teenhome", "Wines", "Fruits", "Meat", "Fish", "Sweets", "Gold",
    "NumDealsPurchases", "NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases", "NumWebVisitsMonth",
    "Income_IQR_outlier", "Age_IQR_outlier", "Spent_IQR_outlier",
]
numeric_features = [c for c in numeric_features if c in customers_clean.columns]
categorical_features = [c for c in ["Education_Group", "Living_With"] if c in customers_clean.columns]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", RobustScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

X = preprocessor.fit_transform(customers_clean[numeric_features + categorical_features])
print(f"Model matrix shape after preprocessing: {X.shape}")

In [ ]:
# MW-METHOD #07 - PCA explained variance justifies the retained dimensions
pca_full = PCA(random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

pca_report = pd.DataFrame({
    "component": np.arange(1, len(explained) + 1),
    "explained_variance_ratio": explained,
    "cumulative_variance": cumulative,
})
display(pca_report.head(12))

n_components_85 = int(np.argmax(cumulative >= 0.85) + 1)
print(f"Components needed for >=85% variance: {n_components_85}")

plt.figure(figsize=(8, 4))
plt.plot(pca_report["component"], pca_report["cumulative_variance"], marker="o")
plt.axhline(0.85, linestyle="--")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA cumulative explained variance")
plt.show()

n_components_model = min(max(3, n_components_85), X.shape[1], X.shape[0] - 1)
pca_model = PCA(n_components=n_components_model, random_state=RANDOM_STATE)
X_reduced = pca_model.fit_transform(X)

pca_3d = PCA(n_components=3, random_state=RANDOM_STATE).fit_transform(X)
print(f"Reduced matrix for clustering: {X_reduced.shape}; 3D projection retained for visualisation: {pca_3d.shape}")

In [ ]:
# MW-METHOD #08 - Cluster-number validation with multiple metrics
validation_rows = []
for k in range(2, 9):
    candidates = {
        "KMeans": KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE),
        "AgglomerativeWard": AgglomerativeClustering(n_clusters=k, linkage="ward"),
    }
    for model_name, model in candidates.items():
        labels = model.fit_predict(X_reduced)
        validation_rows.append({
            "model": model_name,
            "k": k,
            "silhouette": silhouette_score(X_reduced, labels),
            "calinski_harabasz": calinski_harabasz_score(X_reduced, labels),
            "davies_bouldin": davies_bouldin_score(X_reduced, labels),
        })

cluster_validation = pd.DataFrame(validation_rows)
display(cluster_validation.sort_values(["silhouette", "calinski_harabasz"], ascending=[False, False]).head(10))

plt.figure(figsize=(8, 4))
sns.lineplot(data=cluster_validation, x="k", y="silhouette", hue="model", marker="o")
plt.title("Cluster validation: silhouette score")
plt.show()

best_row = cluster_validation.sort_values(["silhouette", "calinski_harabasz"], ascending=[False, False]).iloc[0]
print("Selected configuration from validation metrics:")
display(best_row.to_frame().T)

In [ ]:
# MW-METHOD #09 - Fit final clustering model based on validation result
selected_k = int(best_row["k"])
selected_model_name = best_row["model"]

if selected_model_name == "KMeans":
    final_clusterer = KMeans(n_clusters=selected_k, n_init=50, random_state=RANDOM_STATE)
else:
    final_clusterer = AgglomerativeClustering(n_clusters=selected_k, linkage="ward")

labels = final_clusterer.fit_predict(X_reduced)
customers_profile = customers_clean.copy()
customers_profile["Cluster"] = labels
customers_profile["PC1"] = pca_3d[:, 0]
customers_profile["PC2"] = pca_3d[:, 1]
customers_profile["PC3"] = pca_3d[:, 2]

print(customers_profile["Cluster"].value_counts().sort_index())

plt.figure(figsize=(7, 5))
sns.scatterplot(data=customers_profile, x="PC1", y="PC2", hue="Cluster", palette="tab10", alpha=0.75)
plt.title("Customer clusters projected onto the first two PCs")
plt.show()

In [ ]:
# MW-METHOD #10 - Cluster profiling with descriptive statistics
profile_numeric = [
    "Income", "Spent", "Age", "Recency", "Customer_For_Days", "Children", "Family_Size",
    "NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases", "NumWebVisitsMonth", "Total_Promos", "Response"
]
profile_numeric = [c for c in profile_numeric if c in customers_profile.columns]
cluster_profile = customers_profile.groupby("Cluster")[profile_numeric].agg(["count", "mean", "median"]).round(2)
display(cluster_profile)

# A compact persona table helps interpret the final cluster structure.
persona = customers_profile.groupby("Cluster").agg(
    n_customers=("Cluster", "size"),
    median_income=("Income", "median"),
    median_spent=("Spent", "median"),
    median_age=("Age", "median"),
    parent_rate=("Is_Parent", "mean"),
    web_purchases=("NumWebPurchases", "median"),
    store_purchases=("NumStorePurchases", "median"),
    campaign_response_rate=("Response", "mean"),
).round(2)
display(persona)

In [ ]:
# MW-METHOD #11 - Formal tests: are cluster differences statistically meaningful?
# Numerical variables: Kruskal-Wallis (non-parametric, robust to non-normality).
# Categorical variables: chi-square test with Cramer's V effect size.
num_tests = []
for col in profile_numeric:
    groups = [g[col].dropna() for _, g in customers_profile.groupby("Cluster")]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) >= 2:
        stat, p_value = kruskal(*groups)
        n = sum(len(g) for g in groups)
        k = len(groups)
        epsilon_sq = max((stat - k + 1) / (n - k), 0) if n > k else np.nan
        num_tests.append({"feature": col, "test": "Kruskal-Wallis", "statistic": stat, "p_value": p_value, "effect_size_epsilon_sq": epsilon_sq})

num_tests = pd.DataFrame(num_tests)
if not num_tests.empty:
    num_tests["p_adj_bh"] = benjamini_hochberg(num_tests["p_value"])
    display(num_tests.sort_values("p_adj_bh"))

cat_tests = []
for col in ["Education_Group", "Living_With", "Is_Parent", "Response"]:
    if col in customers_profile.columns:
        table = pd.crosstab(customers_profile["Cluster"], customers_profile[col])
        if table.shape[0] > 1 and table.shape[1] > 1:
            chi2, p_value, dof, expected = chi2_contingency(table)
            cat_tests.append({"feature": col, "test": "chi-square", "chi2": chi2, "p_value": p_value, "cramers_v": cramers_v(table)})

cat_tests = pd.DataFrame(cat_tests)
if not cat_tests.empty:
    cat_tests["p_adj_bh"] = benjamini_hochberg(cat_tests["p_value"])
    display(cat_tests.sort_values("p_adj_bh"))

## Conclusion for Part 1

The workflow frames customer segmentation as a quantitative modelling problem rather than a purely visual clustering exercise. Missing values are audited and imputed, outliers are treated with IQR-based winsorisation and retained flags, categorical variables use one-hot encoding, PCA is selected using explained variance, cluster count is evaluated with silhouette, Calinski-Harabasz and Davies-Bouldin metrics, and final profiles are supported with Kruskal-Wallis and chi-square tests plus effect sizes.

Cluster labels should be assigned only after inspecting the executed `persona` table and confirming that the statistical profile supports the interpretation.